# Improved Global PCA: Validation-Grounded Scoring Ablations

**CS 663 — Digital Image Processing | MVTec AD**

This notebook preserves the lecture-based Global PCA method: flatten each grayscale image, subtract the normal-training mean, find principal directions through the sample-space Gram matrix, project, reconstruct, and use squared reconstruction residuals as anomaly evidence.

It does not replace PCA with a deep model or patch-wise PCA. It tests two interpretable refinements to the residual-based detector: (1) mild Gaussian smoothing of the residual map to reduce isolated pixel noise, and (2) top-k mean pooling for the image-level score instead of relying only on the single largest pixel. It also compares retained-variance settings (90%, 95%, 98%).

**Leakage rule:** fit PCA using `train/good` only; calibrate each image threshold using normal validation images only; use official test images and masks only for final reporting. Do not choose settings by repeatedly inspecting test metrics. Treat the test table as a pre-declared ablation comparison, not validation-based hyperparameter tuning.

## 1. Setup

Set `RUN_ALL_CATEGORIES=True` for all 15 categories. For a quick check, keep it `False` and use `bottle`. This notebook reads the same repository pipeline and dataset path as the baseline notebook.

In [ ]:
from pathlib import Path
import os, sys, subprocess, time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, label as connected_components
from scipy.integrate import trapezoid
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    REPO = Path('/content/CS-663')
    if REPO.exists():
        subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin','main'], check=True)
    else:
        subprocess.run(['git','clone','https://github.com/pranitas-cloud/CS-663.git',str(REPO)], check=True)
else:
    REPO = Path.cwd()
    if not (REPO/'src').is_dir(): raise RuntimeError('Run from the CS-663 repository root.')
os.chdir(REPO)
if str(REPO.resolve()) not in sys.path: sys.path.insert(0,str(REPO.resolve()))
from src.data.pipelines import load_config, build_dataset, stack_images

CONFIG = load_config('configs/default.yaml')
DATA_ROOT = Path(os.environ.get('MVTEC_DATA_ROOT', REPO/'data'/'mvtec_anomaly_detection')).expanduser()
if not DATA_ROOT.is_absolute(): DATA_ROOT = REPO/DATA_ROOT
CONFIG['data']['root'] = str(DATA_ROOT)
SEED = int(CONFIG.get('seed',42))
EXPECTED_CATEGORIES = ['bottle','cable','capsule','carpet','grid','hazelnut','leather','metal_nut','pill','screw','tile','toothbrush','transistor','wood','zipper']
RUN_ALL_CATEGORIES = False
SMOKE_TEST_CATEGORY = 'bottle'
RETAINED_VARIANCES = [0.90, 0.95, 0.98]
SMOOTHING_SIGMA = 1.0   # pixels at the 256 x 256 working resolution
TOP_FRACTIONS = [0.01, 0.05]  # fraction of pixels averaged for image-level score
np.random.seed(SEED); torch.manual_seed(SEED)
if not DATA_ROOT.is_dir(): raise FileNotFoundError(f'MVTec folder not found: {DATA_ROOT}')
available = {p.name for p in DATA_ROOT.iterdir() if p.is_dir() and (p/'train'/'good').is_dir()}
if RUN_ALL_CATEGORIES:
    missing = sorted(set(EXPECTED_CATEGORIES)-available)
    if missing: raise FileNotFoundError(f'Missing categories: {missing}')
    CATEGORIES = EXPECTED_CATEGORIES
else:
    if SMOKE_TEST_CATEGORY not in available: raise FileNotFoundError(SMOKE_TEST_CATEGORY)
    CATEGORIES = [SMOKE_TEST_CATEGORY]
print('Repository:',REPO,'\nDataset:',DATA_ROOT,'\nCategories:',CATEGORIES)


## 2. Global PCA model (same mathematical method)

The eigendecomposition is computed once per category. The three retained-variance settings select different prefixes of the same ordered principal directions; this avoids fitting three separate models. The principal directions are still obtained from the sample-space Gram matrix, which is equivalent to the covariance-eigenvector formulation but is smaller when the number of training images is much less than the number of pixels.

In [ ]:
class GlobalPCAFull:
    def fit(self, training_images):
        x = np.asarray(training_images,dtype=np.float64)
        if x.ndim != 4 or x.shape[0] < 2: raise ValueError('Expected at least two NCHW training images.')
        if not np.isfinite(x).all(): raise ValueError('Non-finite training values.')
        self.image_shape = x.shape[1:]
        flat = x.reshape(x.shape[0],-1)
        self.mean = flat.mean(axis=0)
        xc = flat-self.mean
        gram = xc@xc.T
        vals, vecs = np.linalg.eigh(gram)
        order = np.argsort(vals)[::-1]
        vals = np.maximum(vals[order],0.0); vecs = vecs[:,order]
        valid = vals > max(vals[0]*1e-10,1e-12)
        vals,vecs = vals[valid],vecs[:,valid]
        if len(vals)==0: raise ValueError('No measurable training variance.')
        self.components = (vecs.T@xc)/np.sqrt(vals[:,None])
        self.variance_ratios = vals/vals.sum()
        self.cumulative_variance = np.cumsum(self.variance_ratios)
        return self

    def component_count(self, retained_variance):
        return int(np.searchsorted(self.cumulative_variance,retained_variance)+1)

    def anomaly_map(self, images, retained_variance):
        x = np.asarray(images,dtype=np.float64)
        if x.shape[1:] != self.image_shape: raise ValueError('Image shape differs from training shape.')
        k = self.component_count(retained_variance)
        flat = x.reshape(x.shape[0],-1)
        centered = flat-self.mean
        coeff = centered@self.components[:k].T
        reconstructed = coeff@self.components[:k]+self.mean
        residual = (flat-reconstructed).reshape(x.shape)
        return np.mean(residual**2,axis=1).astype(np.float32),k,float(self.cumulative_variance[k-1])

def get_outputs(dataset, model, retained_variance, sigma):
    labels=[]; maps=[]
    for i in range(len(dataset)):
        item=dataset[i]
        arr=item['image'].unsqueeze(0).cpu().numpy()
        amap,_,_=model.anomaly_map(arr,retained_variance)
        amap=amap[0]
        if sigma>0: amap=gaussian_filter(amap,sigma=sigma)
        labels.append(int(item['label'])); maps.append(amap)
    return np.asarray(labels,dtype=np.uint8),np.stack(maps)

def image_score(anomaly_map, mode):
    flat=np.asarray(anomaly_map).reshape(-1)
    if mode=='max': return float(flat.max())
    fraction=float(mode.split('_')[1])
    count=max(1,int(np.ceil(fraction*len(flat))))
    return float(np.mean(np.partition(flat,-count)[-count:]))

def compute_aupro(masks, maps, max_fpr=0.30, n_thresholds=128):
    masks=np.asarray(masks,dtype=bool); maps=np.asarray(maps,dtype=np.float64)
    regions=[]
    for mask in masks:
        lab,n=connected_components(mask)
        for j in range(1,n+1):
            r=(lab==j)
            if r.any(): regions.append(r)
    negatives=int((~masks).sum())
    if not regions or negatives==0: return float('nan')
    thresholds=np.unique(np.quantile(maps,np.linspace(0,1,n_thresholds)))[::-1]
    fprs=[0.0]; pros=[0.0]
    for t in thresholds:
        pred=maps>=t; fpr=np.logical_and(pred,~masks).sum()/negatives
        if fpr>max_fpr: break
        pros.append(float(np.mean([np.logical_and(pred,r).sum()/r.sum() for r in regions])))
        fprs.append(float(fpr))
    x=np.asarray(fprs); y=np.asarray(pros); order=np.argsort(x); x=x[order]; y=y[order]
    ux=np.unique(x); uy=np.asarray([y[x==v].max() for v in ux])
    inside=ux<max_fpr; x=ux[inside]; y=uy[inside]
    if len(x)==0 or x[0]!=0: x=np.insert(x,0,0.); y=np.insert(y,0,0.)
    y_end=float(np.interp(max_fpr,ux,uy)); x=np.append(x,max_fpr); y=np.append(y,y_end)
    return float(np.clip(trapezoid(y,x)/max_fpr,0,1))


## 3. Run pre-declared ablations

For each variance setting, compare three image-level scores: maximum residual (baseline), mean of the highest-scoring 1% of pixels, and mean of the highest-scoring 5% of pixels. Pixel localization is evaluated on the residual map itself. Gaussian smoothing is applied with a fixed small sigma of 1 working-resolution pixel; this is a simple spatial denoising operation, not a learned model.

Each image threshold is the 95th percentile of that variant's **normal validation image scores**. The test split is used only after threshold calibration.

In [ ]:
all_results=[]
preview=None
for category in CATEGORIES:
    print('\n===',category,'==='); start=time.perf_counter()
    train=build_dataset(CONFIG,'pca','train',category)
    val=build_dataset(CONFIG,'pca','val',category)
    test=build_dataset(CONFIG,'pca','test',category)
    assert all(s[2]=='good' for s in train.samples) and all(s[2]=='good' for s in val.samples)
    model=GlobalPCAFull().fit(stack_images(train).cpu().numpy())
    test_labels=np.asarray([int(test[i]['label']) for i in range(len(test))],dtype=np.uint8)
    test_masks=np.stack([test[i]['mask'].squeeze(0).cpu().numpy().astype(bool) for i in range(len(test))])
    for variance in RETAINED_VARIANCES:
        val_labels,val_maps=get_outputs(val,model,variance,SMOOTHING_SIGMA)
        labels,test_maps=get_outputs(test,model,variance,SMOOTHING_SIGMA)
        assert np.array_equal(labels,test_labels)
        if np.unique(test_labels).size!=2: raise ValueError(f'{category}: test must contain both classes')
        k=model.component_count(variance); actual_var=float(model.cumulative_variance[k-1])
        for score_mode in ['max','top_0.01','top_0.05']:
            val_scores=np.asarray([image_score(m,score_mode) for m in val_maps])
            threshold=float(np.quantile(val_scores,0.95,method='higher'))
            test_scores=np.asarray([image_score(m,score_mode) for m in test_maps])
            pred=(test_scores>=threshold).astype(np.uint8)
            fm=test_masks.reshape(-1).astype(np.uint8); fs=test_maps.reshape(-1)
            row={'category':category,'retained_variance_target':variance,'retained_variance_actual':actual_var,'components':k,'smoothing_sigma':SMOOTHING_SIGMA,'image_score':score_mode,'image_auroc':roc_auc_score(test_labels,test_scores),'image_ap':average_precision_score(test_labels,test_scores),'image_precision':precision_score(test_labels,pred,zero_division=0),'image_recall':recall_score(test_labels,pred,zero_division=0),'image_f1':f1_score(test_labels,pred,zero_division=0),'pixel_auroc':roc_auc_score(fm,fs),'pixel_ap':average_precision_score(fm,fs),'aupro_30pct_fpr':compute_aupro(test_masks,test_maps),'validation_threshold':threshold,'validation_images':len(val),'test_images':len(test)}
            all_results.append(row)
        print(f'variance={variance:.2f}, components={k}, actual={actual_var:.4f} complete')
    if preview is None:
        sample=val[0]['image'].unsqueeze(0).cpu().numpy(); raw,_,_=model.anomaly_map(sample,0.95)
        preview=(sample[0,0],raw[0],gaussian_filter(raw[0],sigma=SMOOTHING_SIGMA))
    print('elapsed seconds:',round(time.perf_counter()-start,1))

results=pd.DataFrame(all_results)
display(results.round(4))
summary=(results.groupby(['retained_variance_target','image_score'])[['image_auroc','image_ap','image_f1','pixel_auroc','pixel_ap','aupro_30pct_fpr']].mean().reset_index())
print('Macro averages across categories included in this run (only a true 15-category macro when RUN_ALL_CATEGORIES=True):')
display(summary.round(4))
out=Path('results/tables'); out.mkdir(parents=True,exist_ok=True)
results.to_csv(out/'global_pca_improved_ablation_categories.csv',index=False)
summary.to_csv(out/'global_pca_improved_ablation_summary.csv',index=False)
print('Saved:',out/'global_pca_improved_ablation_categories.csv')
print('Saved:',out/'global_pca_improved_ablation_summary.csv')
if preview is not None:
    fig,ax=plt.subplots(1,3,figsize=(12,4))
    ax[0].imshow(preview[0],cmap='gray'); ax[0].set_title('Normal validation image')
    ax[1].imshow(preview[1],cmap='inferno'); ax[1].set_title('Raw squared residual')
    ax[2].imshow(preview[2],cmap='inferno'); ax[2].set_title('Smoothed residual (sigma=1)')
    for a in ax:a.axis('off')
    plt.tight_layout(); plt.show()


## 4. How to defend the changes in the viva

**What remains unchanged:** Global PCA still learns the normal-image subspace from mean-centred training images. The principal directions are eigenvectors of the covariance matrix (computed efficiently through the equivalent sample-space Gram matrix). Reconstruction error remains the anomaly signal.

**Change A — residual-map smoothing:** Squared reconstruction error can contain isolated high-valued pixels caused by resampling, texture, or small reconstruction fluctuations. A small Gaussian filter averages nearby residuals. This is a standard spatial smoothing operation; it can improve spatial coherence, but may blur tiny defects. That is why sigma is fixed and its effect is reported rather than assumed to help.

**Change B — top-k image scoring:** The maximum pixel score is sensitive to one noisy pixel. Averaging the highest 1% or 5% of residual pixels still focuses on the most anomalous region, but is less sensitive to a single outlier. The trade-off is that very small defects may be diluted, so compare all three scores.

**Change C — retained variance ablation:** 90%, 95%, and 98% retained variance change the dimensionality of the normal subspace. Keeping more components generally reconstructs normal images more accurately, but can also reconstruct defect structure and reduce anomaly contrast. There is no universally best value; report the comparison.

**Important reporting rule:** do not claim an improvement unless the results show it. If a variant improves image AUROC but worsens pixel AUROC/AUPRO, describe that trade-off. The 95th-percentile threshold is calibrated on normal validation images; it is not optimized using test labels.

**Limitations:** this remains a global, position-sensitive model. It may struggle with defects that occur at variable locations, normal texture variation, or small local changes. Smoothing and top-k pooling do not solve those structural limitations; they are modest, interpretable refinements.